In [1]:
import os
import re
import time
import pickle
import requests
from pathlib import Path
from typing import List, Optional, Tuple
from urllib.parse import quote

import numpy as np
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from huggingface_hub import list_repo_files
from tqdm.auto import tqdm
from langchain_ollama import ChatOllama

# ---------- ДАТАСЕТ И ДОКУМЕНТЫ ----------
DATASET_NAME       = "parlorsky/legal-rag-benchmark-ru"
SAMPLE_DOCS_REPO   = "parlorsky/legal-rag-benchmark-ru"
SAMPLE_DOCS_FOLDER = "sample_docs"

# ---------- EMBEDDINGS ----------
EMBEDDER_NAME = "intfloat/multilingual-e5-small"
# e5 требует префиксы: "passage: " для индексации, "query: " для запроса

# ---------- ЧАНКИНГ ----------
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150

# ---------- RETRIEVAL ----------
TOP_K         = 10
SIM_THRESHOLD = 0.835    # откалиброван: out 0.78–0.83, in 0.845–0.93
KEEP_RATIO    = 0.85     # отсекаем чанки ниже 85% от лучшего
MAX_CONTEXT_CHARS = 8000 # безопасно для num_ctx=6144 (~13k символов бюджета)

# ---------- OLLAMA / LLM ----------
OLLAMA_MODEL          = "qwen2.5:3b"
OLLAMA_NUM_CTX        = 6144   # ⚠️ было 2048 — мало, резало system prompt
OLLAMA_NUM_GPU        = 0      # ⚠️ стабильно только на CPU для MX250
OLLAMA_REPEAT_PENALTY = 1.1    # мягкий штраф, не ломает слова
OLLAMA_REPEAT_LAST_N  = 128
OLLAMA_NUM_PREDICT    = 600    # было 400 — обрывало длинные ответы

REFUSAL_PHRASE = ("Я не могу ответить на этот вопрос, так как в предоставленных "
                  "документах нет необходимой информации.")

In [2]:
ds = load_dataset(DATASET_NAME, split="test")
print(f"Загружено вопросов: {len(ds)}")
print("Пример:", ds[0])

Загружено вопросов: 25
Пример: {'id': 'def_01', 'question': 'Что такое кредитная организация согласно закону о банках?', 'expected_answer': 'Кредитная организация - юридическое лицо, которое для извлечения прибыли как основной цели своей деятельности на основании специального разрешения (лицензии) Центрального банка Российской Федерации (Банка России) имеет право осуществлять банковские операции, предусмотренные Федеральным законом. Кредитная организация образуется на основе любой формы собственности как хозяйственное общество.', 'question_type': 'definition', 'difficulty': 'easy', 'source_docs': ['Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельности'], 'relevant_entities': ['кредитная организация', 'банк', 'лицензия', 'Банк России']}


In [3]:
def sanitize_filename(name: str) -> str:
    """Заменяет недопустимые в Windows символы на подчёркивания."""
    name = re.sub(r'[<>:"/\\|?*]', '_', name).strip('. ')
    return name[:200] if len(name) > 200 else name

docs_dir = Path("sample_docs")
docs_dir.mkdir(exist_ok=True)

all_files = list_repo_files(SAMPLE_DOCS_REPO, repo_type="dataset")
doc_files = [f for f in all_files
             if f.startswith(SAMPLE_DOCS_FOLDER) and f.endswith(".txt")]
print(f"Найдено файлов: {len(doc_files)}")

BASE_URL = f"https://huggingface.co/datasets/{SAMPLE_DOCS_REPO}/resolve/main/"
for file_path in tqdm(doc_files, desc="Скачивание"):
    local_path = docs_dir / sanitize_filename(Path(file_path).name)
    if local_path.exists():
        continue
    url = BASE_URL + quote(file_path, safe='/')
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        local_path.write_bytes(r.content)
    except Exception as e:
        print(f"❌ {Path(file_path).name}: {e}")

print("Готово.")

Найдено файлов: 12


Скачивание:   0%|          | 0/12 [00:00<?, ?it/s]

Готово.


In [4]:
documents, doc_names = [], []

for txt_file in sorted(docs_dir.glob("*.txt")):
    text = txt_file.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 100:
        documents.append(text)
        doc_names.append(txt_file.name)

print(f"Загружено документов: {len(documents)}")
for name, text in zip(doc_names, documents):
    print(f"  {name}: {len(text):,} символов")

Загружено документов: 12
  265_Фонд_микрокредитования_Ирк_обл.txt: 66,154 символов
  Инструкция ЦБ РФ от 28.06.2017 N 180-И Об обязательных нормативах банков.txt: 457,741 символов
  Положение Банка России № 590-П (06.07.2022).txt: 173,538 символов
  Постановление Правительства РФ от 11 октября 2014 г N 1044 Об утверждении Програ.txt: 168,938 символов
  Указание-Банка-России-от-08.10.2018-N-4927-У-_ред.-от-22.09.txt: 2,907,774 символов
  Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельности.txt: 486,135 символов
  Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом строительст.txt: 359,928 символов
  Федеральный закон от 31 декабря 2014 г N 488 ФЗ О промышленной политике в Россий.txt: 162,321 символов
  Федеральный закон № 218-ФЗ _О публично-правовой компании «Фонд защиты прав граждан — участников долевого строительства»_ .txt: 177,676 символов
  • Федеральный закон № 224-ФЗ «О государственно-частном партнёрстве (ГЧП)» – регулирует финансирование инфр

In [5]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks, chunk_metadata = [], []
for doc_name, doc_text in zip(doc_names, documents):
    for i, chunk in enumerate(chunk_text(doc_text)):
        all_chunks.append(chunk)
        chunk_metadata.append({"doc_name": doc_name, "chunk_index": i})

print(f"Всего чанков: {len(all_chunks)}")

Всего чанков: 8746


In [6]:
EMB_CACHE_PATH = Path("chunk_embeddings.pkl")

# 1. Модель загружается ВСЕГДА (нужна и для индексации, и для поиска)
embedder = SentenceTransformer(EMBEDDER_NAME)
print(f"Модель: {EMBEDDER_NAME} | устройство: {embedder.device}")

# 2. Проверка кэша
cached = None
if EMB_CACHE_PATH.exists():
    with open(EMB_CACHE_PATH, "rb") as f:
        cached = pickle.load(f)
    ok = (cached.get("num_chunks") == len(all_chunks)
          and cached.get("embedder_name") == EMBEDDER_NAME
          and cached.get("uses_prefix") is True)
    if ok:
        print(f"✅ Кэш валиден: shape={cached['embeddings'].shape}")
    else:
        print("⚠️ Кэш не подходит — пересчитываем.")
        cached = None

# 3. Считаем или берём из кэша
if cached is None:
    # e5-модели требуют префикс "passage: " для индексируемых текстов
    texts = [f"passage: {c}" for c in all_chunks]

    probe = min(32, len(texts))
    t0 = time.time()
    _ = embedder.encode(texts[:probe], normalize_embeddings=True, batch_size=32)
    per = (time.time() - t0) / probe
    print(f"⏱️ ~{per*1000:.1f} мс/чанк → оценка: ~{per*len(texts)/60:.1f} мин")

    t0 = time.time()
    chunk_embeddings = embedder.encode(
        texts, normalize_embeddings=True, convert_to_numpy=True,
        batch_size=32, show_progress_bar=True,
    )
    print(f"✅ Готово за {(time.time()-t0)/60:.1f} мин")

    with open(EMB_CACHE_PATH, "wb") as f:
        pickle.dump({
            "num_chunks":    len(all_chunks),
            "embedder_name": EMBEDDER_NAME,
            "uses_prefix":   True,
            "embeddings":    chunk_embeddings,
        }, f)
    print(f"💾 Сохранено в {EMB_CACHE_PATH}")
else:
    chunk_embeddings = cached["embeddings"]

# 4. FAISS
t0 = time.time()
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype("float32"))
print(f"✅ FAISS за {time.time()-t0:.2f} с: {index.ntotal} векторов, "
      f"dim={chunk_embeddings.shape[1]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Модель: intfloat/multilingual-e5-small | устройство: cuda:0
✅ Кэш валиден: shape=(8746, 384)
✅ FAISS за 0.02 с: 8746 векторов, dim=384


In [7]:
def retrieve(query: str, top_k: int = TOP_K,
             threshold: float = SIM_THRESHOLD,
             keep_ratio: float = KEEP_RATIO,
             max_chars: int = MAX_CONTEXT_CHARS
             ) -> Tuple[Optional[List[str]], Optional[List[dict]], float]:
    """
    Возвращает (chunks, metadata, best_score).
    Если best_score < threshold → (None, None, best_score) — сигнал к отказу.
    """
    q_emb = embedder.encode(
        [f"query: {query}"],
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")

    scores, idxs = index.search(q_emb, top_k)
    scores, idxs = scores[0], idxs[0]
    best = float(scores[0])

    if best < threshold:
        return None, None, best

    kept_chunks, kept_meta, total = [], [], 0
    for score, idx in zip(scores, idxs):
        if score < best * keep_ratio:
            continue
        c = all_chunks[idx]
        if total + len(c) > max_chars and kept_chunks:
            break
        kept_chunks.append(c)
        kept_meta.append(chunk_metadata[idx])
        total += len(c)
    return kept_chunks, kept_meta, best

In [8]:
from sentence_transformers import CrossEncoder

NLI_AVAILABLE  = False
_nli_entail_idx = 0
_nli_contradict_idx = 2

try:
    nli_model = CrossEncoder("cointegrated/rubert-base-cased-nli-threeway")
    _labels = nli_model.model.config.id2label
    print(f"✅ NLI загружена. Метки: {_labels}")
    # Определяем индексы entail/contradict автоматически
    _nli_entail_idx = next((i for i, l in _labels.items() if 'entail' in l.lower()), 0)
    _nli_contradict_idx = next((i for i, l in _labels.items() if 'contradict' in l.lower()), 2)
    print(f"   entail_idx={_nli_entail_idx}, contradict_idx={_nli_contradict_idx}")
    NLI_AVAILABLE = True
except Exception as e:
    print(f"⚠️ NLI модель недоступна: {e}")
    print("   Код продолжит работать без семантической проверки.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ NLI загружена. Метки: {0: 'entailment', 1: 'contradiction', 2: 'neutral'}
   entail_idx=0, contradict_idx=1


In [15]:
# ---------- LLM ----------
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0.1,
    num_ctx=OLLAMA_NUM_CTX,
    num_gpu=OLLAMA_NUM_GPU,
    repeat_penalty=OLLAMA_REPEAT_PENALTY,
    repeat_last_n=OLLAMA_REPEAT_LAST_N,
    num_predict=OLLAMA_NUM_PREDICT,
    base_url="http://localhost:11434",
)

# Быстрый тест соединения
try:
    probe = llm.invoke("Ответь одним словом по-русски: работает?").content.strip()
    print(f"✅ Ollama доступна. Тест: {probe!r}")
except Exception as e:
    print(f"❌ Ollama недоступна: {e}")
    print("   Убедитесь, что запущено `ollama serve` и скачана модель.")

# ---------- Системный промпт ----------
SYSTEM_PROMPT = """Ты — юридический ассистент. Отвечай только на русском языке.
Используй ИСКЛЮЧИТЕЛЬНО текст, приведённый ниже в разделе КОНТЕКСТ.
Не используй свои знания. Не додумывай. Не пиши markdown и списки."""


def _build_prompt(query: str, context: str) -> str:
    """
    Инструкция-запрет ставится В КОНЦЕ промпта — там, где модель
    обращает больше внимания при генерации.
    """
    return f"""{SYSTEM_PROMPT}

КОНТЕКСТ:
{context}

ВОПРОС: {query}

ПРАВИЛА ОТВЕТА (обязательны):
1. Ответ должен быть основан ТОЛЬКО на тексте КОНТЕКСТА выше.
2. Если ответа в контексте нет — верни ровно одну фразу:
   «{REFUSAL_PHRASE}»
3. Не используй знания о праве других стран. Не пиши «важно отметить», «в целом».
4. Максимум 3 предложения или 3 пункта.

ОТВЕТ:"""


# ---------- Хелперы ----------
def _looks_like_refusal(answer: str) -> bool:
    """True, если ответ — чистый отказ, а не уточнение в середине."""
    a = answer.strip().lower()
    if len(a) > 300:
        return False
    return ("не могу ответить" in a or "нет необходимой информации" in a) and \
           not any(w in a for w in ["однако", "но ", "тем не менее", "согласно"])


def strip_markdown(text: str) -> str:
    """Убирает **bold**, *italic*, `code`."""
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    return text


def dedupe_repeated_lines(text: str, max_dupes: int = 1) -> str:
    """Убирает повторяющиеся строки (защита от loop)."""
    seen = {}
    result = []
    for line in text.split("\n"):
        key = line.strip().lower()
        if not key:
            result.append(line)
            continue
        seen[key] = seen.get(key, 0) + 1
        if seen[key] <= max_dupes:
            result.append(line)
    return "\n".join(result)


def answer_is_clean(answer: str) -> Tuple[bool, str]:
    """Проверяет отсутствие артефактов: CJK, underscores, фейковых номеров, смеси алфавитов."""
    if _looks_like_refusal(answer):
        return True, ""

    # 1) Китайские / корейские / японские символы
    cjk = re.findall(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]', answer)
    if cjk:
        return False, f"восточноазиатские символы: {''.join(cjk[:10])}"

    # 2) Underscore внутри слова (порча токенизации)
    if re.search(r'[A-Za-zА-Яа-яЁё]+_[A-Za-zА-Яа-яЁё]+', answer):
        return False, "underscore внутри слов"

    # 3) Фейковые многоуровневые номера строк: 2.4.1.1.2.1 (4+ уровня)
    if re.search(r'\b\d+(?:\.\d+){3,}\b', answer):
        return False, "фейковые многоуровневые номера"

    # 4) Смешение латиницы и кириллицы внутри одного слова
    if re.search(r'[А-Яа-яЁё]+[A-Za-z]+[А-Яа-яЁё]', answer) or \
       re.search(r'[A-Za-z]+[А-Яа-яЁё]+[A-Za-z]', answer):
        return False, "смешение алфавитов в слове"

    return True, ""


def answer_is_grounded_nli(answer: str, context: str,
                           debug: bool = False) -> Tuple[bool, str]:
    """Семантическая проверка: каждое предложение должно следовать из контекста.

    Ослабленные пороги:
      - reject только если contradiction ЯВНО высокая (c > 0.5) И значительно
        превышает entailment (c > e + 0.2)
      - или если entailment совсем нулевой (e < 0.05) при ненулевой contradiction (c > 0.2)
      - короткие ответы (1 предложение) не проверяем — там шум NLI особенно велик
    """
    if not NLI_AVAILABLE or _looks_like_refusal(answer):
        return True, ""

    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer)
             if len(s.strip()) > 15]
    if len(sents) < 2:            # ⚠️ короткие ответы не проверяем
        if debug:
            print(f"  NLI: пропуск (предложений < 2: {len(sents)})")
        return True, ""

    pairs = [(context, s) for s in sents]
    scores = nli_model.predict(pairs)

    bad = []
    for s, sc in zip(sents, scores):
        e = float(sc[_nli_entail_idx])
        c = float(sc[_nli_contradict_idx])
        if debug:
            print(f"  NLI: entail={e:.3f}, contradict={c:.3f} | {s[:100]}...")

        # ⚠️ ОСЛАБЛЕННЫЕ условия отказа
        is_bad = (c > 0.5 and c > e + 0.2) or (e < 0.05 and c > 0.2)
        if is_bad:
            bad.append(s)

    if bad:
        return False, f"{len(bad)}/{len(sents)} предложений не подтверждены"
    return True, ""

✅ Ollama доступна. Тест: 'Да'


In [16]:
def generate_answer(query: str) -> dict:
    """
    Возвращает dict: {answer, refused, best_score, sources, refusal_reason}
    Guardrails:
      1. Порог retrieval (SIM_THRESHOLD)
      2. Промпт с запретом + refusal-фраза
      3. _looks_like_refusal на ответ LLM
      4. answer_is_clean (артефакты)
      5. answer_is_grounded_nli (семантика)
    """
    chunks, meta, best_score = retrieve(query)

    # Guardrail #1: низкая релевантность → отказ без LLM
    if chunks is None:
        return {"answer": REFUSAL_PHRASE, "refused": True,
                "best_score": best_score, "sources": [],
                "refusal_reason": "low_relevance"}

    context = "\n\n---\n\n".join(chunks)
    prompt = _build_prompt(query, context)

    try:
        raw = llm.invoke(prompt).content.strip()
    except Exception as e:
        return {"answer": f"[Ошибка LLM: {e}]", "refused": False,
                "best_score": best_score, "sources": meta,
                "refusal_reason": "llm_error"}

    # Пост-обработка: markdown + дедуп строк
    answer = strip_markdown(raw)
    answer = dedupe_repeated_lines(answer)

    # Guardrail #3: LLM вернула отказ
    if _looks_like_refusal(answer):
        return {"answer": REFUSAL_PHRASE, "refused": True,
                "best_score": best_score, "sources": meta,
                "refusal_reason": "llm_refused"}

    # Guardrail #4: артефакты
    ok, reason = answer_is_clean(answer)
    if not ok:
        return {"answer": REFUSAL_PHRASE, "refused": True,
                "best_score": best_score, "sources": meta,
                "refusal_reason": f"dirty:{reason}"}

    # Guardrail #5: семантическая проверка
    ok, reason = answer_is_grounded_nli(answer, context)
    if not ok:
        return {"answer": REFUSAL_PHRASE, "refused": True,
                "best_score": best_score, "sources": meta,
                "refusal_reason": f"ungrounded:{reason}"}

    return {"answer": answer, "refused": False,
            "best_score": best_score, "sources": meta,
            "refusal_reason": ""}

In [17]:
print("=" * 80)
print("IN-SCOPE ВОПРОСЫ")
print("=" * 80)

for i in range(5):
    q = ds[i]["question"]
    res = generate_answer(q)
    print(f"\nВОПРОС: {q}")
    print(f"ОТВЕТ:  {res['answer']}")
    status = "ОТКАЗ" if res["refused"] else "ОТВЕТ"
    reason = f" ({res['refusal_reason']})" if res["refusal_reason"] else ""
    print(f"Статус: {status}{reason} | score={res['best_score']:.3f}")

IN-SCOPE ВОПРОСЫ

ВОПРОС: Что такое кредитная организация согласно закону о банках?
ОТВЕТ:  Кредитная организация - юридическое лицо, которое для извлечения прибыли как основной цели своей деятельности на основании специального разрешения (лицензии) Центрального банка Российской Федерации (Банка России) имеет право осуществлять банковские операции, предусмотренные настоящим Федеральным законом. Кредитная организация образуется на основе любой формы собственности как хозяйственное общество.
Статус: ОТВЕТ | score=0.903

ВОПРОС: Чем отличается банк с универсальной лицензией от банка с базовой лицензией?
ОТВЕТ:  Банк с универсальной лицензией имеет право осуществлять банковские операции, указанные в части первой статьи 5 настоящего Федерального закона, без учета ограничений, установленных статьей 5.1 настоящего Федерального закона. Банк с базовой лицензией же имеет право осуществлять банковские операции, указанные в части первой статьи 5 настоящего Федерального закона, с учетом ограничений

In [18]:
out_of_scope = [
    "Какая погода сегодня в Москве?",
    "Как приготовить борщ?",
    "Кто выиграл чемпионат мира по футболу 2022?",
    "Посоветуй хороший сериал на вечер.",
    "Сколько будет 2 + 2?",
]

print("=" * 80)
print("OUT-OF-SCOPE (ожидаются отказы)")
print("=" * 80)

correct = 0
for q in out_of_scope:
    res = generate_answer(q)
    ok = res["refused"]
    correct += int(ok)
    mark = "✅" if ok else "❌"
    print(f"{mark} | {q} | score={res['best_score']:.3f} | {res['refusal_reason'] or 'ОТВЕТ!'}")

print(f"\nИтог: {correct}/{len(out_of_scope)} корректных отказов")

OUT-OF-SCOPE (ожидаются отказы)
✅ | Какая погода сегодня в Москве? | score=0.779 | low_relevance
✅ | Как приготовить борщ? | score=0.799 | low_relevance
✅ | Кто выиграл чемпионат мира по футболу 2022? | score=0.779 | low_relevance
✅ | Посоветуй хороший сериал на вечер. | score=0.808 | low_relevance
✅ | Сколько будет 2 + 2? | score=0.825 | low_relevance

Итог: 5/5 корректных отказов


In [19]:
def evaluate(n_in: int = 10):
    in_refusals = 0
    print(f"\nIn-scope (n={n_in})...")
    for i in tqdm(range(min(n_in, len(ds)))):
        res = generate_answer(ds[i]["question"])
        if res["refused"]:
            in_refusals += 1

    out_ok = 0
    print("Out-of-scope...")
    for q in out_of_scope:
        if generate_answer(q)["refused"]:
            out_ok += 1

    print(f"\n{'='*60}")
    print(f"In-scope  отказов: {in_refusals}/{min(n_in, len(ds))} "
          f"(приемлемо: 0–3 — остальные должны быть точными)")
    print(f"Out-of-scope корректных отказов: {out_ok}/{len(out_of_scope)} "
          f"(должно быть {len(out_of_scope)}/{len(out_of_scope)})")
    print(f"{'='*60}")
    print("\nℹ️ Если in-scope отказов выросло до 3–5 — это ПРАВИЛЬНО: NLI отсекает")
    print("   галлюцинации 3B-модели. Лучше 5 точных ответов, чем 10 сомнительных.")

evaluate(n_in=10)


In-scope (n=10)...


  0%|          | 0/10 [00:00<?, ?it/s]

Out-of-scope...

In-scope  отказов: 1/10 (приемлемо: 0–3 — остальные должны быть точными)
Out-of-scope корректных отказов: 5/5 (должно быть 5/5)

ℹ️ Если in-scope отказов выросло до 3–5 — это ПРАВИЛЬНО: NLI отсекает
   галлюцинации 3B-модели. Лучше 5 точных ответов, чем 10 сомнительных.


In [20]:
in_scores, out_scores = [], []

for i in tqdm(range(min(20, len(ds))), desc="in-scope"):
    _, _, s = retrieve(ds[i]["question"], threshold=0.0)
    in_scores.append(s)

for q in out_of_scope:
    _, _, s = retrieve(q, threshold=0.0)
    out_scores.append(s)

print(f"\nIn-scope:     min={min(in_scores):.3f}  max={max(in_scores):.3f}  "
      f"mean={np.mean(in_scores):.3f}")
print(f"Out-of-scope: min={min(out_scores):.3f}  max={max(out_scores):.3f}  "
      f"mean={np.mean(out_scores):.3f}")

gap = min(in_scores) - max(out_scores)
print(f"\nЗазор: {gap:.3f}")
if gap > 0:
    print(f"✅ Рекомендуемый порог: {max(out_scores) + gap/2:.3f}")
else:
    print("⚠️ Распределения пересекаются — одного порога мало, нужен NLI")

in-scope:   0%|          | 0/20 [00:00<?, ?it/s]


In-scope:     min=0.845  max=0.933  mean=0.884
Out-of-scope: min=0.779  max=0.825  mean=0.798

Зазор: 0.020
✅ Рекомендуемый порог: 0.835
